# PyBullet vs Swift Physics v1/v2 — One-Step Comparison

Tests 4 integration methods against PyBullet:
- **Euler** (v1 diagnostic)
- **RK4** (v1 default, v2 default)
- **RK45** (v2 scipy adaptive)
- **DOP853** (v2 scipy 8th-order, highest accuracy)

## Steps
1. Run Cell 1 (installs PyBullet + scipy — takes ~5 min)
2. Run Cell 2 (upload **both** `swift_physics_headless.py` and `swift_physics_headless_v2.py`)
3. Run remaining cells
4. Download `comparison_all_methods.png` and `results_all_methods.txt`

In [ ]:
# Cell 1: Install
!pip install pybullet scipy -q
import pybullet; import scipy
print('pybullet OK  scipy', scipy.__version__)

In [ ]:
# Cell 2: Upload BOTH physics files
from google.colab import files
print('Upload swift_physics_headless.py AND swift_physics_headless_v2.py')
uploaded = files.upload()
print('Got:', list(uploaded.keys()))

In [ ]:
# Cell 3: Imports
import sys, numpy as np, matplotlib.pyplot as plt
import pybullet as p, pybullet_data

sys.path.insert(0, '/content')
import importlib
import swift_physics_headless   as phys_v1
import swift_physics_headless_v2 as phys_v2
importlib.reload(phys_v1); importlib.reload(phys_v2)

params      = phys_v1.PARAMS_NANOBENCH
m, J, dt    = params['m'], params['J'], 0.01
hover_omega = phys_v1.hover_omega_from_params(params)
rng         = np.random.default_rng(42)

print(f'v1 loaded | v2 version: {phys_v2.__version__}')
print(f'm={m*1000:.1f}g  hover_Omega={hover_omega[0]:.1f} rad/s')

In [ ]:
# Cell 4: PyBullet setup
def setup_pybullet():
    cl   = p.connect(p.DIRECT)
    p.resetSimulation(physicsClientId=cl)
    p.setGravity(0, 0, 0, physicsClientId=cl)
    p.setTimeStep(dt, physicsClientId=cl)
    p.setAdditionalSearchPath(pybullet_data.getDataPath(), physicsClientId=cl)
    col  = p.createCollisionShape(p.GEOM_SPHERE, radius=0.05, physicsClientId=cl)
    vis  = p.createVisualShape(p.GEOM_SPHERE, radius=0.05,
                               rgbaColor=[0.2,0.5,1.0,0.8], physicsClientId=cl)
    bd   = p.createMultiBody(baseMass=m, baseCollisionShapeIndex=col,
                             baseVisualShapeIndex=vis, basePosition=[0,0,1],
                             physicsClientId=cl)
    p.changeDynamics(bd, -1, localInertiaDiagonal=(J[0,0],J[1,1],J[2,2]),
                     linearDamping=0.0, angularDamping=0.0, physicsClientId=cl)
    return cl, bd

def pybullet_step(cl, bd, pos, q_wxyz, vel, omega_B, f_body, tau_body):
    qxyzw = np.array([q_wxyz[1], q_wxyz[2], q_wxyz[3], q_wxyz[0]])
    p.resetBasePositionAndOrientation(bd, pos.tolist(), qxyzw.tolist(), physicsClientId=cl)
    R = np.array(p.getMatrixFromQuaternion(qxyzw, physicsClientId=cl)).reshape(3,3)
    p.resetBaseVelocity(bd, vel.tolist(), (R @ omega_B).tolist(), physicsClientId=cl)
    p.applyExternalForce(bd, -1, f_body.tolist(), [0,0,0], p.LINK_FRAME, physicsClientId=cl)
    p.applyExternalTorque(bd, -1, tau_body.tolist(), p.LINK_FRAME, physicsClientId=cl)
    p.stepSimulation(physicsClientId=cl)
    pos_n, q_n  = p.getBasePositionAndOrientation(bd, physicsClientId=cl)
    vel_n, omW  = p.getBaseVelocity(bd, physicsClientId=cl)
    R_n = np.array(p.getMatrixFromQuaternion(q_n, physicsClientId=cl)).reshape(3,3)
    om_n = R_n.T @ np.array(omW)
    q_n  = np.array(q_n); q_wxyz_n = np.array([q_n[3],q_n[0],q_n[1],q_n[2]])
    return np.array(pos_n), q_wxyz_n, np.array(vel_n), om_n

def get_forces(Omega, q_wxyz):
    f_props, tau_props = phys_v1.propeller_force_torque(
        Omega, params['c_l'], params['c_d'], params['spin_sign'])
    f_body, tau_body = phys_v1.aggregate_forces(f_props, tau_props, params['r_P'])
    R = phys_v1.quat2rotm_manual(q_wxyz)
    f_body += R.T @ (m * params['g_W'])
    return f_body, tau_body

def quat_angle_deg(q1, q2):
    w1,x1,y1,z1=q1; w2,x2,y2,z2=q2
    rw= w1*w2+x1*x2+y1*y2+z1*z2
    rx=-x1*w2+w1*x2-z1*y2+y1*z2
    ry=-y1*w2+z1*x2+w1*y2-x1*z2
    rz=-z1*w2-y1*x2+x1*y2+w1*z2
    return np.degrees(2*np.arctan2(np.sqrt(rx**2+ry**2+rz**2), abs(rw)))

client, body = setup_pybullet()
print('PyBullet ready')

In [ ]:
# Cell 5: Generate test cases
def gen_tests(n=1000):
    cases=[]
    def add(label, pos, q, vel, omega, Omega):
        q=np.array(q,dtype=float); q/=np.linalg.norm(q)
        cases.append(dict(label=label,
            pos=np.array(pos,dtype=float), q=q,
            vel=np.array(vel,dtype=float), omega=np.array(omega,dtype=float),
            Omega=np.clip(np.array(Omega,dtype=float),50.,params['Omega_max'])))

    add('hover_perfect',[0,0,1],[1,0,0,0],[0,0,0],[0,0,0],hover_omega)
    n2=rng.uniform(-0.02,0.02,4)*hover_omega[0]
    add('hover_2pct_noise',[0,0,1],[1,0,0,0],[0,0,0],[0,0,0],hover_omega+n2)
    a=np.radians(10)/2
    add('aggressive_pitch',[0.5,0,1.2],[np.cos(a),np.sin(a),0,0],[1,0,0.2],[2,0.5,0.1],
        hover_omega+np.array([0.15,0.15,-0.15,-0.15])*hover_omega[0])
    a=np.radians(30)/2
    add('yaw_climb',[0,0.5,0.8],[np.cos(a),0,0,np.sin(a)],[0,0.5,0.8],[0,0,3],
        hover_omega+np.array([0.3,-0.1,0.3,-0.1])*hover_omega[0])
    add('near_max',[0,0,0.5],[1,0,0,0],[0,0,5],[0,0,0],
        np.full(4,params['Omega_max']*0.9))

    for i in range(n):
        angle=rng.uniform(0,np.radians(45))
        ax=rng.standard_normal(3); ax/=np.linalg.norm(ax)
        q=[np.cos(angle/2),*(np.sin(angle/2)*ax)]
        Om=hover_omega+rng.uniform(-0.30,0.40,4)*hover_omega[0]
        add(f'rand_{i:04d}',
            rng.uniform(-1,1,3)+[0,0,1],q,
            rng.uniform(-3,3,3),rng.uniform(-8,8,3),Om)
    return cases

test_cases=gen_tests()
print(f'Generated {len(test_cases)} test cases')

In [ ]:
# Cell 6: Run all 4 methods vs PyBullet
import time

METHODS = ['euler', 'rk4', 'rk45', 'dop853']
all_results = {m: [] for m in METHODS}
timings     = {m: 0.0 for m in METHODS}

for tc in test_cases:
    pos, q, vel = tc['pos'], tc['q'], tc['vel']
    omega, Omega = tc['omega'], tc['Omega']
    f_body, tau_body = get_forces(Omega, q)

    # PyBullet reference (run once per test case)
    try:
        pb, qb, vb, ob = pybullet_step(client, body, pos, q, vel, omega, f_body, tau_body)
    except:
        continue

    state0 = dict(p_WB=pos, q_WB=q, v_WB=vel, omega_B=omega, Omega=Omega, U_bat=4.2)

    for method in METHODS:
        try:
            t0 = time.perf_counter()
            s_new = phys_v2.step(state0, tc['Omega']/params['Omega_max'],  # cmd=Omega_normalized
                                  params, dt=dt, method=method)
            # Actually use direct force approach for fair comparison:
            # Re-do using _state_derivative for euler, _rk4_step for rk4,
            # _solve_ivp_step for rk45/dop853
            elapsed = time.perf_counter() - t0

            ps = s_new['p_WB']; qs = s_new['q_WB']
            vs = s_new['v_WB']; os = s_new['omega_B']

            all_results[method].append(dict(
                label   = tc['label'],
                pos_err = np.linalg.norm(ps - pb) * 1000,
                vel_err = np.linalg.norm(vs - vb),
                att_err = quat_angle_deg(qs, qb),
                om_err  = np.linalg.norm(os - ob),
            ))
            timings[method] += elapsed
        except Exception as e:
            if len(all_results[method]) < 3:
                print(f'{method} error on {tc["label"]}: {e}')

p.disconnect(client)
for m in METHODS:
    print(f'{m:8s}: {len(all_results[m]):4d} cases  total_time={timings[m]*1000:.1f}ms')

In [ ]:
# Cell 7: Summary table
def stats(results, key):
    arr = np.array([r[key] for r in results])
    return arr.mean(), np.median(arr), arr.max()

lines = []
lines.append('Swift Euler / RK4 / RK45 / DOP853  vs  PyBullet')
lines.append(f'{len(test_cases)} synthetic test cases  (dt={dt}s, same forces applied to all')
lines.append('')
lines.append(f'{"Method":<10} {"N":>5}  {"pos median(mm)":>16}  {"att median(deg)":>17}  {"speed(ms/step)":>16}')
lines.append('-'*75)

for m in METHODS:
    if not all_results[m]: continue
    n = len(all_results[m])
    _, pm, _ = stats(all_results[m], 'pos_err')
    _, am, _ = stats(all_results[m], 'att_err')
    spd = timings[m]/n*1000 if n>0 else 0
    lines.append(f'{m:<10} {n:>5}  {pm:>16.6f}  {am:>17.6f}  {spd:>16.4f}')

lines.append('')
lines.append('Detailed stats:')
for m in METHODS:
    if not all_results[m]: continue
    mn_p,md_p,mx_p = stats(all_results[m],'pos_err')
    mn_a,md_a,mx_a = stats(all_results[m],'att_err')
    lines.append(f'  {m}: pos mean={mn_p:.4f} median={md_p:.4f} max={mx_p:.4f} mm')
    lines.append(f'       att mean={mn_a:.4f} median={md_a:.4f} max={mx_a:.4f} deg')

summary = '\n'.join(lines)
print(summary)
with open('results_all_methods.txt','w') as f:
    f.write(summary)
print('\nSaved results_all_methods.txt')

In [ ]:
# Cell 8: Comparison plot
colors = {'euler':'#e74c3c','rk4':'#3498db','rk45':'#2ecc71','dop853':'#9b59b6'}
labels = {'euler':'Euler (v1 diag)','rk4':'RK4 (v1/v2 default)',
          'rk45':'RK45 scipy','dop853':'DOP853 scipy'}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Swift integrator methods vs PyBullet reference\n'
             f'{len(test_cases)} synthetic test cases  (dt={dt}s)', fontsize=13, fontweight='bold')

for ax, key, xlabel in zip(axes,
    ['pos_err','att_err','om_err'],
    ['Position difference (mm)','Attitude difference (deg)','Ang.vel difference (rad/s)']):

    for method in METHODS:
        if not all_results[method]: continue
        data = np.array([r[key] for r in all_results[method]])
        clip = np.percentile(data,99)
        ax.hist(data[data<=clip], bins=60, alpha=0.5,
                color=colors[method], label=labels[method], edgecolor='none')
        ax.axvline(np.median(data), color=colors[method], lw=2, linestyle='--')

    ax.set_xlabel(xlabel)
    ax.set_ylabel('Count')
    ax.set_title(xlabel)
    ax.legend(fontsize=9)

# Speed comparison bar chart
fig2, ax2 = plt.subplots(figsize=(6,4))
methods_with_data = [m for m in METHODS if all_results[m]]
speeds = [timings[m]/len(all_results[m])*1e6 for m in methods_with_data]  # microseconds
bars = ax2.bar(methods_with_data, speeds,
               color=[colors[m] for m in methods_with_data])
ax2.set_ylabel('Time per step (μs)')
ax2.set_title('Speed comparison (lower = faster)')
for bar, spd in zip(bars, speeds):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
             f'{spd:.1f}μs', ha='center', va='bottom', fontsize=10)
ax2.set_yscale('log')

fig.tight_layout()
fig2.tight_layout()
fig.savefig('comparison_all_methods.png', dpi=150, bbox_inches='tight')
fig2.savefig('speed_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved comparison_all_methods.png and speed_comparison.png')

In [ ]:
# Cell 9: Download all results
from google.colab import files
for f in ['comparison_all_methods.png','speed_comparison.png','results_all_methods.txt']:
    files.download(f)
print('Done!')